# HARPS grouping V2

Groups the HARPS spectra by star using the target name **and** the coordinates (SIMBAD).
V1 used the reduced name only; V2 checks every name against the catalogue position.

1. Clean the name (V1 sanitise, reduce, normalise) and bin non-stellar targets (Sun, planets, calibrations...)
2. Build query units: (reduced name, catalogue J2000 position `TEL TARG ALPHA/DELTA`), or the pointing if missing
3. SIMBAD cone search (30") around every unit, then all identifiers of the candidates
4. Names that match no candidate are resolved directly in SIMBAD
5. Per spectrum: name match checked against the proper-motion-propagated position, else cone rule
6. Key = `GaiaDR3 <source_id>` (else `SIMBAD <main_id>`), consistency pass, fibre epoch tag

All network calls live in the cells marked `QUERY`. They cache to `<data_dir>/grouping/v2/cache/*.csv`
and only ask for what is missing, so re-running is cheap and everything after them runs offline.

Input: `state.sqlite` from `scripts/harps_fetch.py`. Output: `<data_dir>/grouping/v2/`.

In [ ]:
'''
Updated 29-10-2026

Form groups for every individual target (HARPS), hybrid name + coordinates
'''
import io
import json
import math
import re
import sqlite3
import time
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from astropy.io.votable import from_table, writeto
from astropy.table import Table
from astropy.time import Time


def find_repo(start=Path.cwd()):
    '''Walk up from the notebook until config/harps.json is found.'''
    for folder in [start, *start.parents]:
        if (folder / 'config' / 'harps.json').exists():
            return folder
    raise FileNotFoundError('config/harps.json not found above ' + str(start))


config = json.loads((find_repo() / 'config' / 'harps.json').read_text())
GROUP = config['grouping']
DATA_DIR = Path(config['data_dir'])
DB_PATH = DATA_DIR / 'state.sqlite'
OUT_DIR = DATA_DIR / 'grouping' / 'v2'
CACHE_DIR = OUT_DIR / 'cache'
CACHE_DIR.mkdir(parents=True, exist_ok=True)

TAP_URLS = GROUP['simbad_tap_urls']
FIBRE_MJD = Time(GROUP['fibre_upgrade_date']).mjd  # 2015-05-29 -> 57171
MJD_J2000 = 51544.5

In [ ]:
### Load metadata ###
# Spectra downloaded and covering both Ca II K and H (same selection as V1)
db = sqlite3.connect(f'file:{DB_PATH.as_posix()}?mode=ro', uri=True)  # read-only: safe while the download runs
metadata = pd.read_sql_query(
    "SELECT dp_id, target_name, s_ra, s_dec, t_min, snr, proposal_id, file "
    "FROM spectra WHERE status = 'done' AND covers_K = 1 AND covers_H = 1",
    db,
)

# Catalogue position and proper motion of the target from the FITS header (not a V1 column)
HEADER_KEYS = {
    'HIERARCH ESO TEL TARG ALPHA': 'targ_alpha',
    'HIERARCH ESO TEL TARG DELTA': 'targ_delta',
    'HIERARCH ESO TEL TARG PMA': 'targ_pma',
    'HIERARCH ESO TEL TARG PMD': 'targ_pmd',
    'HIERARCH ESO TEL TARG EPOCH': 'targ_epoch',
}
header_rows = []
for dp_id, header_json in db.execute(
        "SELECT dp_id, header_json FROM spectra WHERE status = 'done' AND covers_K = 1 AND covers_H = 1"):
    header = json.loads(header_json) if header_json else {}
    header_rows.append((dp_id, *[header.get(key) for key in HEADER_KEYS]))
db.close()

header_df = pd.DataFrame(header_rows, columns=['dp_id', *HEADER_KEYS.values()]).astype(
    {col: float for col in HEADER_KEYS.values()})
metadata = metadata.merge(header_df, on='dp_id', how='left')

print(f'Loaded metadata: {len(metadata)} spectra, {metadata["target_name"].nunique()} raw target names')

In [ ]:
### Sanitise + reduce names (identical to V1) ###

def exceptions(name):
    '''Catalogue-specific formatting of a target name.'''

    # Case for BD catalogue
    if name.startswith('BD'):
        return name

    # Case for ex. CD-234-2432
    if name.startswith('CD'):
        return name[:3] + name[3:].replace('-', '')

    # Case for J###-## and G##-##
    if len(name) > 1 and name[0] in ('J', 'G') and name[1].isnumeric():
        return name

    # Case for CPD-###
    if name.startswith('CPD'):
        return name

    # Case for LP catalogue
    if len(name) > 3 and name.startswith('LP') and name[2].isnumeric():
        return name

    # Case for SERAM catalogue
    if len(name) > 5 and name.startswith('SERAM'):
        return name[:7].replace('-', '') + name[7:]

    # Case of SW catalogue
    if name.startswith('SW'):
        return name

    # Case for TYC catalogue
    if name.startswith('TYC'):
        return name[:5].replace('-', '') + name[5:]

    return name.replace('-', ' ')


def sanitise(header):
    '''Archive name -> sanitised name.'''
    # getting rid of 'A_' in example A_HD45637 (every name starting with a letter up to F + '_')
    if len(header) > 1 and header[0].lower() in 'abcdef' and header[1] == '_':
        header = header[2:]
    return exceptions(header).replace('_', ' ')


def reduce(sanitised):
    '''Sanitised name -> reduced name (lowercase, no spaces).'''
    return sanitised.lower().replace(' ', '')


metadata.insert(1, 'Sanitised', metadata['target_name'].fillna('').map(sanitise))
metadata.insert(2, 'Reduced', metadata['Sanitised'].map(reduce))

print(f'Reduced names: {metadata["Reduced"].nunique()} unique')

In [ ]:
### Alias normalization (identical to V1) ###
# Maps non-canonical reduced names → canonical form.
# Keys = aliases confirmed in the old (UVES) data; values = preferred name.
_ALIAS_MAP = {
    'betapic':      'hd39060',
    'betapictoris': 'hd39060',
    'spica':        'alphavir',
    'tetaql':       'thetaaql',
    '49cet':        '49ceti',
    'alphalyr':     'vega',
    'hd216956':     'fomalhaut',
    'betelgeuse':   'hd39801',
    'hd143275':     'deltasco',
}
_HD_ZERO  = re.compile(r'^(hd)0+(\d+)(.*)$')
_HIP_ZERO = re.compile(r'^(hip)0+(\d+)$')
_CS_ZERO  = re.compile(r'^(cs\d{5})0(\d{3,4})$')
_CD_D     = re.compile(r'^(cd-\d+)d(\d+)$')


def normalize_reduced(name):
    '''Return the canonical reduced name, resolving known aliases and catalog formatting.'''
    if name in _ALIAS_MAP:
        return _ALIAS_MAP[name]
    m = _HD_ZERO.match(name)
    if m:
        return m.group(1) + m.group(2) + m.group(3)
    m = _HIP_ZERO.match(name)
    if m:
        return m.group(1) + m.group(2)
    if name.startswith('he~'):
        return 'he' + name[3:]
    m = _CS_ZERO.match(name)
    if m:
        return m.group(1) + m.group(2)
    m = _CD_D.match(name)
    if m:
        return m.group(1) + m.group(2)
    return name


before_unique = metadata['Reduced'].nunique()
metadata['Reduced'] = metadata['Reduced'].map(normalize_reduced)
after_unique = metadata['Reduced'].nunique()
print(f'Alias normalization: {before_unique} → {after_unique} unique names ({before_unique - after_unique} merged)')

In [ ]:
### Non-stellar targets ###
# Sun, sky, planets, minor bodies, comets and calibration frames are not queried (regexes in config)
NONSTELLAR = re.compile('|'.join(f'(?:{p})' for p in GROUP['nonstellar_patterns']))
SUN = re.compile(GROUP['sun_pattern'])


def nonstellar_id(reduced):
    '''Reduced name -> group key if non-stellar, else None. All Sun-like feeds share one group.'''
    if not NONSTELLAR.search(reduced):
        return None
    return 'nonstellar:SUN' if SUN.search(reduced) else f'nonstellar:{reduced}'


metadata['nonstellar_id'] = metadata['Reduced'].map(nonstellar_id)
is_nonstellar = metadata['nonstellar_id'].notna()

print(f'Non-stellar spectra: {is_nonstellar.sum()} in {metadata.loc[is_nonstellar, "nonstellar_id"].nunique()} groups')
print(metadata.loc[is_nonstellar, 'nonstellar_id'].value_counts().head(10))

In [ ]:
### Geometry helpers ###

def sexagesimal_to_deg(value, is_ra):
    '''ESO packed sexagesimal (HHMMSS.s / ±DDMMSS.s) -> degrees. NaN if missing or malformed.'''
    if value is None or not np.isfinite(value):
        return np.nan
    negative = math.copysign(1.0, value) < 0
    a = round(abs(value), 4)
    d = int(a // 10000)
    m = int((a - d * 10000) // 100)
    s = round(a - d * 10000 - m * 100, 4)
    if m >= 60 or s >= 60:
        return np.nan
    deg = d + m / 60 + s / 3600
    if is_ra:
        return deg * 15 if (not negative and d < 24) else np.nan
    return (-deg if negative else deg) if deg <= 90 else np.nan


def sep_arcsec_np(ra1, dec1, ra2, dec2):
    '''Angular separation in arcsec (vectorised, haversine).'''
    ra1, dec1, ra2, dec2 = map(np.radians, (ra1, dec1, ra2, dec2))
    a = np.sin((dec2 - dec1) / 2) ** 2 + np.cos(dec1) * np.cos(dec2) * np.sin((ra2 - ra1) / 2) ** 2
    return np.degrees(2 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))) * 3600


def sep_arcsec(ra1, dec1, ra2, dec2):
    '''Angular separation in arcsec (scalars, haversine).'''
    ra1, dec1, ra2, dec2 = map(math.radians, (ra1, dec1, ra2, dec2))
    a = math.sin((dec2 - dec1) / 2) ** 2 + math.cos(dec1) * math.cos(dec2) * math.sin((ra2 - ra1) / 2) ** 2
    return math.degrees(2 * math.asin(math.sqrt(min(max(a, 0.0), 1.0)))) * 3600


def propagate(ra, dec, pmra, pmdec, t_mjd):
    '''Move a J2000 position to t_mjd. pmra includes cos(dec), mas/yr (SIMBAD convention). Linear.'''
    pmra = 0.0 if pmra is None or not math.isfinite(pmra) else pmra
    pmdec = 0.0 if pmdec is None or not math.isfinite(pmdec) else pmdec
    dt = (t_mjd - MJD_J2000) / 365.25 if t_mjd is not None and math.isfinite(t_mjd) else 0.0
    new_dec = dec + pmdec * dt / 3.6e6
    new_ra = ra + pmra * dt / 3.6e6 / max(math.cos(math.radians(dec)), 1e-6)
    return new_ra % 360.0, new_dec

In [ ]:
### Target position from the header + query units ###
# TEL TARG ALPHA/DELTA = catalogue J2000 position of the target, PMA/PMD in arcsec/yr (incl. cos dec).
# s_ra/s_dec = telescope pointing at the date of observation.
metadata['targ_ra'] = [sexagesimal_to_deg(v, True) for v in metadata['targ_alpha']]
metadata['targ_dec'] = [sexagesimal_to_deg(v, False) for v in metadata['targ_delta']]
# the (0, 0) placeholder is not a position
metadata.loc[(metadata['targ_alpha'] == 0) & (metadata['targ_delta'] == 0), ['targ_ra', 'targ_dec']] = np.nan

# Sanity check: the target position moved with its own proper motion must land near the pointing
dt_yr = (metadata['t_min'] - MJD_J2000) / 365.25
cosd = np.cos(np.radians(metadata['targ_dec'])).clip(lower=1e-6)
targ_ra_obs = metadata['targ_ra'] + metadata['targ_pma'].fillna(0) * dt_yr / 3600 / cosd
targ_dec_obs = metadata['targ_dec'] + metadata['targ_pmd'].fillna(0) * dt_yr / 3600
targ_vs_pointing = sep_arcsec_np(targ_ra_obs, targ_dec_obs, metadata['s_ra'], metadata['s_dec'])
bad_targ = targ_vs_pointing > GROUP['targ_pointing_max_arcsec']
bad_targ &= ~is_nonstellar  # the Sun has ephemeris coordinates in degrees, irrelevant here
metadata['targ_ignored'] = bad_targ
metadata.loc[bad_targ, ['targ_ra', 'targ_dec']] = np.nan

stellar_meta = ~is_nonstellar
has_targ = metadata['targ_ra'].notna() & metadata['targ_dec'].notna()
print(f'Stellar spectra with TARG position: {(has_targ & stellar_meta).sum()} / {stellar_meta.sum()}')
print(f'TARG ignored (far from pointing): {bad_targ.sum()}')
print(f'TARG epoch not 2000: {(stellar_meta & has_targ & (metadata["targ_epoch"] != 2000)).sum()}')

# Query units: (reduced name, TARG J2000 rounded to 1") or (reduced name, pointing rounded to 5")
r1 = GROUP['unit_round_targ_arcsec']
r5 = GROUP['unit_round_pointing_arcsec']
use_targ = has_targ
ref_ra = metadata['targ_ra'].where(use_targ, metadata['s_ra'])
ref_dec = metadata['targ_dec'].where(use_targ, metadata['s_dec'])
step = np.where(use_targ, r1, r5)
ra_i = np.round(ref_ra * 3600 / step).fillna(-1).astype(int)
dec_i = np.round(ref_dec * 3600 / step).fillna(-1).astype(int)
kind = np.where(use_targ, 'targ', np.where(ref_ra.notna(), 'point', 'nopos'))
metadata['unit_kind'] = kind
metadata['ukey'] = metadata['Reduced'] + '|' + metadata['unit_kind'] + '|' + ra_i.astype(str) + '|' + dec_i.astype(str)
metadata.loc[is_nonstellar, 'ukey'] = ''
metadata['ref_ra'] = ref_ra
metadata['ref_dec'] = ref_dec

units = (metadata[stellar_meta].groupby('ukey')
         .agg(Reduced=('Reduced', 'first'), unit_kind=('unit_kind', 'first'),
              qra=('ref_ra', 'median'), qdec=('ref_dec', 'median'), n_spectra=('dp_id', 'size'))
         .reset_index())
units['unit_id'] = np.arange(len(units))

print(f'Query units: {len(units)} ({units["unit_kind"].value_counts().to_dict()})')
print(f'Unique stellar reduced names: {units["Reduced"].nunique()}')

In [ ]:
### Name canonicalisation (same rules for archive names and SIMBAD identifiers) ###
GREEK = {
    'alf': 'alpha', 'alp': 'alpha', 'alpha': 'alpha', 'bet': 'beta', 'beta': 'beta',
    'gam': 'gamma', 'gamma': 'gamma', 'del': 'delta', 'delta': 'delta',
    'eps': 'epsilon', 'epsilon': 'epsilon', 'zet': 'zeta', 'zeta': 'zeta', 'eta': 'eta',
    'tet': 'theta', 'teta': 'theta', 'the': 'theta', 'theta': 'theta', 'iot': 'iota', 'iota': 'iota',
    'kap': 'kappa', 'kappa': 'kappa', 'lam': 'lambda', 'lambda': 'lambda', 'mu': 'mu', 'nu': 'nu',
    'ksi': 'xi', 'xi': 'xi', 'omi': 'omicron', 'omicron': 'omicron', 'pi': 'pi', 'rho': 'rho',
    'sig': 'sigma', 'sigma': 'sigma', 'tau': 'tau', 'ups': 'upsilon', 'upsilon': 'upsilon',
    'phi': 'phi', 'chi': 'chi', 'khi': 'chi', 'psi': 'psi', 'ome': 'omega', 'omega': 'omega',
}
GREEK_ABBR = {'alpha': 'alf', 'beta': 'bet', 'gamma': 'gam', 'delta': 'del', 'epsilon': 'eps', 'zeta': 'zet',
              'eta': 'eta', 'theta': 'tet', 'iota': 'iot', 'kappa': 'kap', 'lambda': 'lam', 'mu': 'mu',
              'nu': 'nu', 'xi': 'ksi', 'omicron': 'omi', 'pi': 'pi', 'rho': 'rho', 'sigma': 'sig',
              'tau': 'tau', 'upsilon': 'ups', 'phi': 'phi', 'chi': 'chi', 'psi': 'psi', 'omega': 'ome'}

# IAU abbreviation, genitive (lowercase, no spaces)
_CONSTELLATIONS = [
    ('And', 'andromedae'), ('Ant', 'antliae'), ('Aps', 'apodis'), ('Aqr', 'aquarii'), ('Aql', 'aquilae'),
    ('Ara', 'arae'), ('Ari', 'arietis'), ('Aur', 'aurigae'), ('Boo', 'bootis'), ('Cae', 'caeli'),
    ('Cam', 'camelopardalis'), ('Cnc', 'cancri'), ('CVn', 'canumvenaticorum'), ('CMa', 'canismajoris'),
    ('CMi', 'canisminoris'), ('Cap', 'capricorni'), ('Car', 'carinae'), ('Cas', 'cassiopeiae'),
    ('Cen', 'centauri'), ('Cep', 'cephei'), ('Cet', 'ceti'), ('Cha', 'chamaeleontis'), ('Cir', 'circini'),
    ('Col', 'columbae'), ('Com', 'comaeberenices'), ('CrA', 'coronaeaustralis'), ('CrB', 'coronaeborealis'),
    ('Crv', 'corvi'), ('Crt', 'crateris'), ('Cru', 'crucis'), ('Cyg', 'cygni'), ('Del', 'delphini'),
    ('Dor', 'doradus'), ('Dra', 'draconis'), ('Equ', 'equulei'), ('Eri', 'eridani'), ('For', 'fornacis'),
    ('Gem', 'geminorum'), ('Gru', 'gruis'), ('Her', 'herculis'), ('Hor', 'horologii'), ('Hya', 'hydrae'),
    ('Hyi', 'hydri'), ('Ind', 'indi'), ('Lac', 'lacertae'), ('Leo', 'leonis'), ('LMi', 'leonisminoris'),
    ('Lep', 'leporis'), ('Lib', 'librae'), ('Lup', 'lupi'), ('Lyn', 'lyncis'), ('Lyr', 'lyrae'),
    ('Men', 'mensae'), ('Mic', 'microscopii'), ('Mon', 'monocerotis'), ('Mus', 'muscae'), ('Nor', 'normae'),
    ('Oct', 'octantis'), ('Oph', 'ophiuchi'), ('Ori', 'orionis'), ('Pav', 'pavonis'), ('Peg', 'pegasi'),
    ('Per', 'persei'), ('Phe', 'phoenicis'), ('Pic', 'pictoris'), ('Psc', 'piscium'), ('PsA', 'piscisaustrini'),
    ('Pup', 'puppis'), ('Pyx', 'pyxidis'), ('Ret', 'reticuli'), ('Sge', 'sagittae'), ('Sgr', 'sagittarii'),
    ('Sco', 'scorpii'), ('Scl', 'sculptoris'), ('Sct', 'scuti'), ('Ser', 'serpentis'), ('Sex', 'sextantis'),
    ('Tau', 'tauri'), ('Tel', 'telescopii'), ('Tri', 'trianguli'), ('TrA', 'triangulaustralis'),
    ('Tuc', 'tucanae'), ('UMa', 'ursaemajoris'), ('UMi', 'ursaeminoris'), ('Vel', 'velorum'),
    ('Vir', 'virginis'), ('Vol', 'volantis'), ('Vul', 'vulpeculae'),
]
CONS_OFFICIAL = {abbr.lower(): abbr for abbr, _ in _CONSTELLATIONS}
CONS_GENITIVE = {gen: abbr.lower() for abbr, gen in _CONSTELLATIONS}
_GEN_SORTED = sorted(CONS_GENITIVE, key=len, reverse=True)
_GREEK_SORTED = sorted(GREEK, key=len, reverse=True)
_GREEK_RE = re.compile(r'^(' + '|'.join(_GREEK_SORTED) + r')0*(\d?)(.+)$')
_TAIL_RE = re.compile(r'^[a-h]{0,2}$')
_SIMBAD_PREFIX = re.compile(r'^(\*\*|\*|v\*|em\*|name)\s+', re.I)


def greek_parts(reduced):
    '''Reduced name like "alfcena" / "betapictoris" -> (greek, digit, constellation_abbr, tail) or None.'''
    m = _GREEK_RE.match(reduced)
    if not m:
        return None
    greek, digit, rest = GREEK[m.group(1)], m.group(2), m.group(3)
    for gen in _GEN_SORTED:
        if rest.startswith(gen):
            cons, tail = CONS_GENITIVE[gen], rest[len(gen):]
            break
    else:
        cons, tail = rest[:3], rest[3:]
        if cons not in CONS_OFFICIAL:
            return None
    if not _TAIL_RE.match(tail):
        return None
    return greek, digit, cons, tail


def greek_canon(reduced):
    '''Greek abbreviations / full names and constellation names/abbreviations -> one spelling.'''
    parts = greek_parts(reduced)
    return reduced if parts is None else ''.join(parts)


def canon_reduced(reduced):
    '''Reduced archive name -> comparable canonical name (V1 aliases applied before and after the Greek step).'''
    return normalize_reduced(greek_canon(normalize_reduced(reduced)))


def canon_name(identifier):
    '''SIMBAD identifier -> comparable canonical name (same sanitise/reduce rules as the archive names).'''
    identifier = _SIMBAD_PREFIX.sub('', identifier.strip())
    identifier = re.sub(r'^(mu|nu|xi|pi)\.\s*', r'\1 ', identifier)  # SIMBAD: 'nu. Oct', 'mu. Ara'
    return canon_reduced(reduce(sanitise(identifier)))

In [ ]:
### Name resolution helpers (SIMBAD spellings of an archive name) ###
CATALOGUES = {'HD', 'HIP', 'HR', 'GJ', 'GL', 'BD', 'CD', 'CPD', 'TYC', 'LHS', 'LTT', 'SAO', 'HIC', 'LP', 'NLTT',
              'HE', 'CS', 'WASP', 'HAT', 'TOI', 'KOI', 'NGC', 'IC', 'DM', 'EPIC', 'TIC', 'KIC', 'CoRoT'}
SIGNED_CATALOGUES = {'BD', 'CD', 'CPD', 'DM'}
SIMBAD_PREFIXES = ['', '* ', 'V* ', '** ', 'NAME ']


def simbad_variants(raw_name):
    '''Archive name -> likely SIMBAD identifier spellings (ident.id is matched exactly).'''
    base = {raw_name.replace('_', ' ').strip(), sanitise(raw_name).strip()}
    out = set(base)
    for s in base:
        m = re.match(r'^([A-Za-z]+)\s*([-+]?\d.*)$', s)
        if m:
            pre, rest = m.groups()
            if pre.upper() in CATALOGUES:
                pre = pre.upper()
                if pre not in SIGNED_CATALOGUES:
                    rest = rest.lstrip('-+')
                if pre in {'HD', 'HIP', 'HR', 'HIC', 'SAO'}:
                    rest = rest.lstrip('0')
                names = {'GJ': ['GJ', 'Gl'], 'GL': ['GJ', 'Gl']}.get(pre, [pre])
                for name in names:
                    out.add(f'{name} {rest}')
        parts = greek_parts(reduce(s))
        if parts is not None:
            greek, digit, cons, tail = parts
            for d in {digit, digit.zfill(2) if digit else ''}:
                out.add(f'{GREEK_ABBR[greek]}{d} {CONS_OFFICIAL[cons]}' + (f' {tail.upper()}' if tail else ''))
    return sorted(prefix + v for v in out if v for prefix in SIMBAD_PREFIXES)


def unit_name_oids(unit_name, cand_oids, oid_names, otypes=None):
    '''Candidates whose identifiers match the unit name.
    Returns (oids, component_ambiguous). A bare system name (e.g. alphacen) that also has
    components among the candidates (alphacena, alphacenb) resolves to the components.'''
    exact = [o for o in cand_oids if unit_name in oid_names.get(o, ())]
    siblings = [o for o in cand_oids
                if any(len(n) == len(unit_name) + 1 and n.startswith(unit_name) and n[-1] in 'abcde'
                       for n in oid_names.get(o, ()))]
    systems = [o for o in exact if (otypes or {}).get(o) == '**']
    if exact and len(systems) < len(exact):  # the named star itself (HD 102365, not HD 102365B)
        return [o for o in exact if o not in systems], False
    if siblings:
        return siblings, True
    if not exact and re.match(r'^[a-z]+\d+[a-e]$', unit_name):  # catalogue number + component letter
        exact = [o for o in cand_oids if unit_name[:-1] in oid_names.get(o, ())]
    return exact, False

In [ ]:
### SIMBAD TAP client (the only code that touches the network) ###

class TapError(Exception):
    '''ADQL / upload problem reported by the server (retrying will not help).'''


def to_votable_bytes(df):
    buffer = io.BytesIO()
    writeto(from_table(Table.from_pandas(df.reset_index(drop=True))), buffer)
    return buffer.getvalue()


def tap_query(query, upload_name=None, upload_df=None):
    '''Run an ADQL query (optionally with one uploaded table) and return a DataFrame.
    URLs are tried in order; network errors and 5xx are retried with exponential backoff.'''
    data = {'REQUEST': 'doQuery', 'LANG': 'ADQL', 'FORMAT': 'csv', 'QUERY': query}
    files = None
    if upload_df is not None:
        data['UPLOAD'] = f'{upload_name},param:{upload_name}'
        files = {upload_name: (f'{upload_name}.xml', to_votable_bytes(upload_df))}
    last_error = None
    for url in TAP_URLS:
        for attempt in range(GROUP['max_tries']):
            try:
                response = requests.post(url, data=data, files=files, timeout=GROUP['timeout_s'])
            except requests.RequestException as error:
                last_error = f'{url}: {error}'
                time.sleep(2 ** attempt * 2)
                continue
            text = response.text
            if response.status_code == 200 and not text.lstrip().startswith('<'):
                if not text.strip():
                    return pd.DataFrame()
                df = pd.read_csv(io.StringIO(text))
                df.columns = [c.lower() for c in df.columns]
                return df
            if response.status_code >= 500:
                last_error = f'{url}: HTTP {response.status_code}'
                time.sleep(2 ** attempt * 2)
                continue
            raise TapError(f'{url}: HTTP {response.status_code}: {text[:500]}')
        print(f'  giving up on {url}, trying next mirror')
    raise ConnectionError(f'All SIMBAD TAP mirrors failed. Last error: {last_error}')


def read_cache(name):
    path = CACHE_DIR / name
    return pd.read_csv(path, dtype=str, keep_default_na=False) if path.exists() else pd.DataFrame()


def append_cache(name, df):
    path = CACHE_DIR / name
    df.to_csv(path, mode='a', header=not path.exists(), index=False)


# Candidate columns returned by the cone and by the name resolution
CAND_SELECT = ("b.oid, b.main_id, b.ra, b.dec, b.pmra, b.pmdec, b.plx_value, b.otype, b.sp_type, "
               "{flux}, g.id AS gaia")
CAND_JOIN = ("JOIN basic AS b ON {on} {flux_join} "
             "LEFT JOIN ident AS g ON g.oidref = b.oid AND g.id LIKE 'Gaia DR3 %'")
FLUX_VARIANTS = [("f.V AS vmag", "LEFT JOIN allfluxes AS f ON f.oidref = b.oid"),
                 ('99.9 AS vmag', '')]  # fallback if the flux column is rejected (99.9 = unknown, sorts last)


def run_chunks(todo_df, upload_name, make_query, chunk_rows):
    '''Yield the result of every chunk of an upload table, trying the flux variants.'''
    for start in range(0, len(todo_df), chunk_rows):
        chunk = todo_df.iloc[start:start + chunk_rows]
        for flux_select, flux_join in FLUX_VARIANTS:
            try:
                result = tap_query(make_query(flux_select, flux_join), upload_name, chunk)
                break
            except TapError as error:
                print(f'  query variant rejected ({str(error)[:150]}), trying the next one')
        else:
            raise TapError('all query variants rejected')
        print(f'  chunk {start // chunk_rows + 1}/{math.ceil(len(todo_df) / chunk_rows)}: '
              f'{len(chunk)} uploaded, {len(result)} rows back')
        yield chunk, result

In [ ]:
### QUERY 1: SIMBAD cone search around every query unit (network, cached, incremental) ###
cone_done = set(read_cache('cone_done.csv').get('ukey', []))
todo = units[~units['ukey'].isin(cone_done) & units['qra'].notna()]
print(f'Cone search: {len(todo)} units to query, {len(cone_done)} already cached')

if len(todo):
    upload = todo[['unit_id', 'qra', 'qdec']].rename(columns={'unit_id': 'pid'})
    radius = GROUP['cone_search_arcsec'] / 3600

    def cone_query(flux_select, flux_join):
        return ("SELECT p.pid, " + CAND_SELECT.format(flux=flux_select) + " FROM TAP_UPLOAD.pts AS p "
                + CAND_JOIN.format(flux_join=flux_join,
                                   on=f"1 = CONTAINS(POINT('ICRS', b.ra, b.dec), CIRCLE('ICRS', p.qra, p.qdec, {radius}))"))

    id_to_ukey = dict(zip(units['unit_id'], units['ukey']))
    for chunk, result in run_chunks(upload, 'pts', cone_query, GROUP['chunk_rows']):
        if len(result):
            result['ukey'] = result['pid'].map(id_to_ukey)
            result = result.drop(columns='pid').drop_duplicates(['ukey', 'oid'])
            append_cache('cone_candidates.csv', result)
        append_cache('cone_done.csv', pd.DataFrame({'ukey': chunk['pid'].map(id_to_ukey)}))

In [ ]:
### QUERY 2: all identifiers of the candidate objects (network, cached, incremental) ###

def fetch_idents():
    '''Download ident.id for every candidate oid (cone + name resolution) not cached yet.'''
    oids = set(read_cache('cone_candidates.csv').get('oid', [])) | set(read_cache('resolved.csv').get('oid', []))
    have = set(read_cache('idents.csv').get('oid', []))
    todo_oids = sorted(oids - have, key=int)
    print(f'Identifiers: {len(todo_oids)} objects to query, {len(have)} already cached')
    chunk_rows = GROUP['chunk_rows']
    for start in range(0, len(todo_oids), chunk_rows):
        chunk = pd.DataFrame({'oid': np.array(todo_oids[start:start + chunk_rows], dtype=np.int64)})
        result = tap_query("SELECT i.oidref AS oid, i.id FROM TAP_UPLOAD.oids AS o "
                           "JOIN ident AS i ON i.oidref = o.oid", 'oids', chunk)
        print(f'  chunk {start // chunk_rows + 1}/{math.ceil(len(todo_oids) / chunk_rows)}: {len(result)} ids')
        append_cache('idents.csv', result)
        missing = set(chunk['oid'].astype(str)) - set(result['oid'].astype(str))
        if missing:  # objects without identifiers: record them so they are not asked again
            append_cache('idents.csv', pd.DataFrame({'oid': sorted(missing), 'id': ''}))


fetch_idents()

In [ ]:
### Link units to their catalogue candidates (offline) ###

def load_candidates():
    '''Cached SIMBAD candidates -> (cone table, resolved table, oid info, oid -> canonical names).'''
    num = ['ra', 'dec', 'pmra', 'pmdec', 'plx_value', 'vmag']
    cone = read_cache('cone_candidates.csv')
    resolved = read_cache('resolved.csv')
    # planets (listed at their host's position), galaxies, IR/X/radio sources... are never the target star
    cone, resolved = [t[~t['otype'].isin(GROUP['exclude_otypes'])] if len(t) else t for t in (cone, resolved)]
    cols = ['oid', 'main_id', 'ra', 'dec', 'pmra', 'pmdec', 'plx_value', 'otype', 'sp_type', 'vmag', 'gaia']
    allc = pd.concat([t[cols] for t in (cone, resolved) if len(t)], ignore_index=True) if (len(cone) or len(resolved)) \
        else pd.DataFrame(columns=cols)
    allc[num] = allc[num].apply(pd.to_numeric, errors='coerce')
    allc = allc.drop_duplicates('oid').set_index('oid')
    info = {}
    for oid, row in allc.iterrows():
        gaia = row['gaia'].replace('Gaia DR3 ', '').strip() if isinstance(row['gaia'], str) else ''
        info[oid] = dict(oid=oid, main_id=row['main_id'], ra=row['ra'], dec=row['dec'], pmra=row['pmra'],
                         pmdec=row['pmdec'], plx=row['plx_value'], otype=row['otype'], sp_type=row['sp_type'],
                         vmag=row['vmag'], gaia=gaia)
    idents = read_cache('idents.csv')
    oid_names = {}
    for oid, ident in zip(idents.get('oid', []), idents.get('id', [])):
        if ident:
            oid_names.setdefault(oid, set()).add(canon_name(ident))
    return cone, resolved, info, oid_names


def link_units(units, cone, resolved, oid_names):
    '''Per unit: candidate oids, oids matching the unit name, component ambiguity.'''
    cone_by_unit = cone.groupby('ukey')['oid'].apply(list).to_dict() if len(cone) else {}
    resolved_by_name = resolved.groupby('reduced')['oid'].apply(list).to_dict() if len(resolved) else {}
    linked = {}
    for ukey, name in zip(units['ukey'], units['Reduced']):
        cand_oids = list(dict.fromkeys(cone_by_unit.get(ukey, []) + resolved_by_name.get(name, [])))
        otypes = {o: cand_info[o]['otype'] for o in cand_oids if o in cand_info}
        name_oids, ambiguous = unit_name_oids(canon_reduced(name), cand_oids, oid_names, otypes)
        if not name_oids:  # resolved directly by name in SIMBAD
            name_oids = [o for o in resolved_by_name.get(name, [])]
        linked[ukey] = dict(cands=cand_oids, name_oids=set(name_oids), ambiguous=ambiguous)
    return linked


cone, resolved, cand_info, oid_names = load_candidates()
linked = link_units(units, cone, resolved, oid_names)
no_name_match = units[[not linked[k]['name_oids'] for k in units['ukey']]]
print(f'Units with a candidate matching the name: {len(units) - len(no_name_match)} / {len(units)}')
print(f'Units with no name match (names to resolve in SIMBAD): {len(no_name_match)} '
      f'({no_name_match["Reduced"].nunique()} unique names)')

In [ ]:
### QUERY 3: resolve the leftover names directly in SIMBAD (network, cached, incremental) ###
# Joining ident against an uploaded table times out on the mirror, so names go in plain IN (...) lists:
# (a) spellings -> oid, matched back by canonical name; (b) oid -> candidate data, like the cone search.
resolved_done = set(read_cache('resolved_done.csv').get('reduced', []))
leftover = sorted(set(no_name_match['Reduced']) - resolved_done)
print(f'Name resolution: {len(leftover)} names to query, {len(resolved_done)} already cached')


def adql_list(values):
    return ', '.join("'" + str(v).replace("'", "''") + "'" for v in values)


if leftover:
    raw_by_reduced = metadata[metadata['Reduced'].isin(leftover)].groupby('Reduced')['target_name'].unique()
    canon_to_reduced = {}
    for name in leftover:
        for raw in raw_by_reduced[name][:3]:
            for variant in simbad_variants(raw):
                canon_to_reduced.setdefault(canon_name(variant), set()).add(name)
    variants = sorted({v for name in leftover for raw in raw_by_reduced[name][:3] for v in simbad_variants(raw)})
    print(f'  {len(variants)} SIMBAD spellings to try')

    # (a) spellings -> oid (SIMBAD returns its own spelling of the id, matched back by canonical name)
    found = []
    step = GROUP['name_chunk']
    for start in range(0, len(variants), step):
        result = tap_query(f"SELECT i.id, i.oidref AS oid FROM ident AS i "
                           f"WHERE i.id IN ({adql_list(variants[start:start + step])})")
        for ident, oid in zip(result.get('id', []), result.get('oid', [])):
            for name in canon_to_reduced.get(canon_name(ident), ()):
                found.append((name, str(oid)))
        print(f'  chunk {start // step + 1}/{math.ceil(len(variants) / step)}: {len(result)} ids found')
    found = pd.DataFrame(found, columns=['reduced', 'oid']).drop_duplicates()

    # (b) oid -> candidate data (same columns as the cone search)
    oids = sorted(found['oid'].unique(), key=int)
    data = []
    for start in range(0, len(oids), step):
        in_list = ', '.join(oids[start:start + step])
        for flux_select, flux_join in FLUX_VARIANTS:
            try:
                data.append(tap_query("SELECT " + CAND_SELECT.format(flux=flux_select) + " FROM basic AS b "
                                      + flux_join + " LEFT JOIN ident AS g ON g.oidref = b.oid AND g.id LIKE 'Gaia DR3 %' "
                                      f"WHERE b.oid IN ({in_list})"))
                break
            except TapError as error:
                print(f'  query variant rejected ({str(error)[:150]}), trying the next one')
        else:
            raise TapError('all query variants rejected')
    if data:
        data = pd.concat(data, ignore_index=True)
        data['oid'] = data['oid'].astype(str)
        result = found.merge(data, on='oid', how='inner').drop_duplicates(['reduced', 'oid'])
        append_cache('resolved.csv', result)
        print(f'Resolved by name: {result["reduced"].nunique()} / {len(leftover)} names')
    append_cache('resolved_done.csv', pd.DataFrame({'reduced': leftover}))

In [ ]:
### QUERY 4: identifiers of the objects found by name resolution (network, cached, incremental) ###
fetch_idents()

cone, resolved, cand_info, oid_names = load_candidates()
linked = link_units(units, cone, resolved, oid_names)
no_name_match = units[[not linked[k]['name_oids'] for k in units['ukey']]]
print(f'Units still without a name match: {len(no_name_match)} ({no_name_match["Reduced"].nunique()} unique names)')

In [ ]:
### Decision per spectrum ###
CFG = dict(name_match=GROUP['name_match_arcsec'], targ_match=GROUP['targ_match_arcsec'],
           cone_radius=GROUP['cone_search_arcsec'], cone_accept=GROUP['cone_accept_arcsec'],
           tie=GROUP['tie_arcsec'])
NAN = float('nan')


def decide(spec, cands, name_oids, ambiguous, cfg):
    '''Assign one spectrum to one candidate.
    spec: dict(ra, dec, t_min, targ_ra, targ_dec); cands: list of candidate dicts (oid, ra, dec, pmra, pmdec, vmag).
    Returns dict(oid, method, sep, flags, name_main).
    Rules: name match accepted if the candidate moved to t_min is within name_match of the pointing, or
    its J2000 position is within targ_match of the header TARG position. Name found but position failing ->
    flag name_position_mismatch and the cone rule decides. Cone: a single candidate within cone_accept
    is accepted; 2+ within cone_radius -> nearest TARG (or pointing), brightness only as tie-break.'''
    flags, name_main = [], ''
    has_p = math.isfinite(spec['ra']) and math.isfinite(spec['dec'])
    has_t = math.isfinite(spec['targ_ra']) and math.isfinite(spec['targ_dec'])

    rows = []  # (candidate, sep propagated->pointing, sep J2000->TARG)
    for c in cands:
        sp = NAN
        if has_p:
            pra, pdec = propagate(c['ra'], c['dec'], c['pmra'], c['pmdec'], spec['t_min'])
            sp = sep_arcsec(pra, pdec, spec['ra'], spec['dec'])
        st = sep_arcsec(c['ra'], c['dec'], spec['targ_ra'], spec['targ_dec']) if has_t else NAN
        rows.append((c, sp, st))

    def passes(r):
        return r[1] <= cfg['name_match'] or r[2] <= cfg['targ_match']  # NaN compares False

    def rank(r):
        return r[2] if has_t else (r[1] if has_p else 0.0)

    def dist(r):
        return r[1] if has_p else r[2]

    def result(r, method):
        return dict(oid=r[0]['oid'], method=method, sep=r[1], flags=flags, name_main=name_main)

    named = [r for r in rows if r[0]['oid'] in name_oids]
    if named:
        good = [r for r in named if passes(r)]
        if good:
            if ambiguous:
                flags.append('component_ambiguous')
            return result(min(good, key=rank), 'name')
        if not has_p and not has_t:
            flags.append('no_position')
            return result(named[0], 'name')
        flags.append('name_position_mismatch')
        name_main = min(named, key=rank)[0]['main_id']
    else:
        flags.append('name_unmatched')

    near = [r for r in rows if dist(r) <= cfg['cone_radius']]
    if len(near) >= 2:
        near.sort(key=rank)
        tied = [r for r in near if rank(r) - rank(near[0]) <= cfg['tie']]
        best = min(tied, key=lambda r: r[0]['vmag'] if math.isfinite(r[0]['vmag']) else 99.0)
        flags.append('close_pair')
        return result(best, 'cone')
    if len(near) == 1:
        if dist(near[0]) > cfg['cone_accept']:
            flags.append('wide_match')  # beyond the primary radius: pointing scatter, to review
        return result(near[0], 'cone')
    return dict(oid=None, method='unresolved', sep=NAN, flags=flags, name_main=name_main)

In [ ]:
### Decide every stellar spectrum ###
stellar_idx = metadata.index[stellar_meta]
decisions = {}
for row in metadata.loc[stellar_idx, ['ukey', 's_ra', 's_dec', 't_min', 'targ_ra', 'targ_dec']].itertuples():
    link = linked[row.ukey]
    spec = dict(ra=row.s_ra, dec=row.s_dec, t_min=row.t_min, targ_ra=row.targ_ra, targ_dec=row.targ_dec)
    decisions[row.Index] = decide(spec, [cand_info[o] for o in link['cands']], link['name_oids'],
                                  link['ambiguous'], CFG)

dec_df = pd.DataFrame.from_dict(decisions, orient='index')
metadata['oid'] = dec_df['oid']
metadata['method'] = dec_df['method']
metadata['sep_arcsec'] = dec_df['sep']
metadata['name_main'] = dec_df['name_main']
metadata['flag_list'] = dec_df['flags']
metadata.loc[is_nonstellar, 'method'] = 'nonstellar'
metadata['flag_list'] = metadata['flag_list'].map(lambda x: x if isinstance(x, list) else [])
metadata.loc[metadata['targ_ignored'], 'flag_list'] = metadata.loc[metadata['targ_ignored'], 'flag_list'].map(
    lambda x: x + ['targ_ignored'])
print(metadata['method'].value_counts())

In [ ]:
### Canonical key: GaiaDR3 <source_id>, else SIMBAD <main_id> ###
def star_key(oid):
    info = cand_info[oid]
    star_id = f'GaiaDR3 {info["gaia"]}' if info['gaia'] else f'SIMBAD {info["main_id"]}'
    return star_id, info['main_id'], info['gaia']


keys = {oid: star_key(oid) for oid in metadata['oid'].dropna().unique()}
metadata['star_id'] = metadata['oid'].map(lambda o: keys[o][0] if o in keys else None)
metadata['star_label'] = metadata['oid'].map(lambda o: keys[o][1] if o in keys else None)
metadata['gaia_dr3'] = metadata['oid'].map(lambda o: keys[o][2] if o in keys else None)

# Non-stellar and unresolved spectra keep their own groups, kept out of the stellar sample via 'method'
metadata.loc[is_nonstellar, 'star_id'] = metadata.loc[is_nonstellar, 'nonstellar_id']
metadata.loc[is_nonstellar, 'star_label'] = metadata.loc[is_nonstellar, 'nonstellar_id']
unresolved = metadata['method'] == 'unresolved'
metadata.loc[unresolved, 'star_id'] = 'unresolved:' + metadata.loc[unresolved, 'Reduced']
metadata.loc[unresolved, 'star_label'] = metadata.loc[unresolved, 'Reduced']

In [ ]:
### Consistency pass ###
# (a) each reduced name should map to one star
stars_per_name = metadata[stellar_meta & ~unresolved].groupby('Reduced')['star_id'].nunique()
multi_names = stars_per_name[stars_per_name > 1].index
multi_set = set(multi_names)
metadata['flag_list'] = [flags + ['name_multiple_stars'] if red in multi_set else flags
                         for flags, red in zip(metadata['flag_list'], metadata['Reduced'])]
print(f'Reduced names mapped to more than one star: {len(multi_names)}')

# (b) each spectrum should lie within the radius of the propagated position of its star
outlier = (metadata['method'].isin(['name', 'cone'])) & (metadata['sep_arcsec'] > GROUP['name_match_arcsec'])
metadata['flag_list'] = [flags + ['position_outlier'] if out else flags for flags, out in zip(metadata['flag_list'], outlier)]
print(f'Spectra further than {GROUP["name_match_arcsec"]}" from their star (accepted by TARG position): {outlier.sum()}')

In [ ]:
### Fibre epoch tag + flags + group numbers ###
# Fibre change of 2015-05-29 (HARPS): tag only, groups are not split
metadata['epoch_fibre'] = np.where(metadata['t_min'] < FIBRE_MJD, 'pre2015', 'post2015')
metadata['flags'] = metadata['flag_list'].map(lambda x: ';'.join(sorted(set(x))))
metadata = metadata.drop(columns=['flag_list'])

star_to_id = {star: i for i, star in enumerate(sorted(metadata['star_id'].unique()))}
metadata['New Groups'] = metadata['star_id'].map(star_to_id)
print(f'Spectra with at least one flag: {(metadata["flags"] != "").sum()}')

In [ ]:
### Get groups ###
def flag_union(series):
    return ';'.join(sorted({f for cell in series for f in cell.split(';') if f}))


grouped = metadata.groupby('New Groups')
groups = grouped.agg(star_id=('star_id', 'first'), star_label=('star_label', 'first'),
                     gaia_dr3=('gaia_dr3', 'first'), method=('method', lambda s: s.value_counts().index[0]),
                     Nb_Obs=('dp_id', 'size'), n_raw_names=('target_name', 'nunique'),
                     raw_names=('target_name', lambda s: sorted(s.unique())),
                     flags=('flags', flag_union)).rename(columns={'Nb_Obs': 'Nb Obs'}).reset_index()

oid_of_group = grouped['oid'].agg(lambda s: s.dropna().iloc[0] if s.notna().any() else None)
for col, key in [('ra', 'ra'), ('dec', 'dec'), ('pmra', 'pmra'), ('pmdec', 'pmdec'), ('plx', 'plx'),
                 ('sp_type', 'sp_type'), ('V', 'vmag')]:
    groups[col] = [cand_info[o][key] if o is not None and o in cand_info else np.nan
                   for o in oid_of_group.reindex(groups['New Groups'])]

sizes = groups['Nb Obs']
print(f'Groups: {len(groups)}')
print(f'Groups with < 3 spectra: {(sizes < 3).sum()}')
print(f'Groups with < 2 spectra: {(sizes < 2).sum()}')
print(groups.sort_values('Nb Obs', ascending=False).head(10))

In [ ]:
### Conflicts list ###
# One row per (unit, reason); spectra-level flags stay in the metadata
rows = []
flagged = metadata[metadata['flags'] != '']
for (reduced, ukey, flag_str, star_id, label, name_main), sub in flagged.groupby(
        ['Reduced', 'ukey', 'flags', 'star_id', 'star_label', 'name_main'], dropna=False):
    for flag in flag_str.split(';'):
        rows.append(dict(reduced=reduced, unit=ukey, reason=flag, n_spectra=len(sub), star_id=star_id,
                         star_label=label, name_match_main_id=name_main, sep_arcsec_max=sub['sep_arcsec'].max(),
                         example_dp_id=sub['dp_id'].iloc[0], raw_names=';'.join(sorted(sub['target_name'].unique()))))
for name in multi_names:
    sub = metadata[(metadata['Reduced'] == name) & stellar_meta]
    rows.append(dict(reduced=name, unit='', reason='name_multiple_stars', n_spectra=len(sub),
                     star_id=';'.join(sorted(sub['star_id'].unique())), star_label='', name_match_main_id='',
                     sep_arcsec_max=sub['sep_arcsec'].max(), example_dp_id=sub['dp_id'].iloc[0],
                     raw_names=';'.join(sorted(sub['target_name'].unique()))))
conflicts = pd.DataFrame(rows)
print(f'Conflict rows: {len(conflicts)}')
if len(conflicts):
    print(conflicts['reason'].value_counts())

In [ ]:
### Save outputs ###
full_path = OUT_DIR / 'harps_full_metadata_v2.pkl'
groups_path = OUT_DIR / 'harps_groups_v2.pkl'
conflicts_path = OUT_DIR / 'conflicts.csv'

save_cols = ['dp_id', 'target_name', 'Sanitised', 'Reduced', 's_ra', 's_dec', 't_min', 'snr', 'proposal_id', 'file',
             'star_id', 'star_label', 'gaia_dr3', 'method', 'sep_arcsec', 'flags', 'epoch_fibre', 'New Groups']
metadata[save_cols].to_pickle(full_path)
groups.to_pickle(groups_path)
conflicts.to_csv(conflicts_path, index=False)
print(f'Saved: {full_path}')
print(f'Saved: {groups_path}')
print(f'Saved: {conflicts_path}')

In [ ]:
### Summary ###
print(f'Groups: {len(groups)}')
print('Spectra per method:')
print(metadata['method'].value_counts())
print('Spectra per flag:')
print(metadata['flags'].str.split(';').explode().replace('', np.nan).value_counts())
print(f'Groups with < 3 spectra: {(groups["Nb Obs"] < 3).sum()}')
print(f'Groups with < 2 spectra: {(groups["Nb Obs"] < 2).sum()}')
print(f'Unresolved names: {metadata.loc[unresolved, "Reduced"].nunique()} ({unresolved.sum()} spectra)')
print(metadata['epoch_fibre'].value_counts())

In [ ]:
### Optional: offline test of the decision rules (set RUN_TESTS = True) ###
RUN_TESTS = True

if RUN_TESTS:
    t = 55000.0

    def cand(oid, ra, dec, pmra=0.0, pmdec=0.0, vmag=8.0, name='x'):
        return dict(oid=oid, main_id=name, ra=ra, dec=dec, pmra=pmra, pmdec=pmdec, vmag=vmag)

    def spec(ra, dec, targ=None):
        return dict(ra=ra, dec=dec, t_min=t, targ_ra=targ[0] if targ else NAN, targ_dec=targ[1] if targ else NAN)

    off = lambda arcsec: arcsec / 3600
    # packed sexagesimal incl. negative and small values
    assert abs(sexagesimal_to_deg(175748.50, True) - (17 + 57 / 60 + 48.5 / 3600) * 15) < 1e-9
    assert abs(sexagesimal_to_deg(44136.2, False) - (4 + 41 / 60 + 36.2 / 3600)) < 1e-9
    assert abs(sexagesimal_to_deg(-510400.0, False) + (51 + 4 / 60)) < 1e-9
    assert abs(sexagesimal_to_deg(-4530.0, False) + (45 / 60 + 30 / 3600)) < 1e-9
    assert abs(sexagesimal_to_deg(3638.0, True) - (36 / 60 + 38 / 3600) * 15) < 1e-9
    assert math.isnan(sexagesimal_to_deg(126000.0, False)) and math.isnan(sexagesimal_to_deg(250000.0, True))
    # name canonicalisation
    assert canon_name('bet Pic') == canon_name('* bet Pic') == canon_name('beta_Pic') == canon_name('V* beta Pictoris')
    assert canon_name('HD039060') == canon_name('HD 39060') == canon_name('HD-39060')
    assert canon_name('* alf Cen A') == canon_name('alpha Cen A') == canon_name('alphacenA')
    assert canon_name('HD 128620') != canon_name('HD 128621')
    # name match, position OK (high proper motion moves the star 37" in 10 yr: needs propagation)
    pm = 3678.0
    c_a = cand(1, 220.0, -60.8, pmra=-pm, pmdec=0.0)
    ra_t = propagate(220.0, -60.8, -pm, 0.0, t)[0]
    r = decide(spec(ra_t + off(2), -60.8), [c_a], {1}, False, CFG)
    assert r['method'] == 'name' and r['oid'] == 1 and not r['flags']
    assert sep_arcsec(220.0, -60.8, ra_t, -60.8) > 30  # unpropagated would have failed
    # name matches but pointing far away -> mismatch, nothing in cone -> unresolved
    r = decide(spec(10.0, 10.0), [cand(1, 220.0, -60.8)], {1}, False, CFG)
    assert r['method'] == 'unresolved' and 'name_position_mismatch' in r['flags']
    # name matches far away but another star sits at the pointing -> position wins, both recorded
    r = decide(spec(30.0, 5.0), [cand(1, 220.0, -60.8, name='named'), cand(2, 30.0, 5.0 + off(2), name='here')],
               {1}, False, CFG)
    assert r['oid'] == 2 and r['method'] == 'cone' and r['name_main'] == 'named' and 'name_position_mismatch' in r['flags']
    # TARG within 3" of the SIMBAD J2000 accepts the name even if pointing is off by > 30"
    r = decide(spec(30.0 + off(60), 5.0, targ=(30.0 + off(1), 5.0)), [cand(1, 30.0, 5.0)], {1}, False, CFG)
    assert r['method'] == 'name'
    # components: name picks the named one (distinct HD numbers), not the nearer neighbour
    r = decide(spec(50.0, 5.0), [cand(1, 50.0, 5.0 + off(3)), cand(2, 50.0, 5.0 + off(6))], {2}, False, CFG)
    assert r['oid'] == 2 and r['method'] == 'name'
    # cone: single candidate within 10" accepted, flagged name_unmatched
    r = decide(spec(50.0, 5.0), [cand(1, 50.0, 5.0 + off(4))], set(), False, CFG)
    assert r['method'] == 'cone' and r['flags'] == ['name_unmatched']
    # cone: single candidate between 10" and 30" is accepted and flagged
    r = decide(spec(50.0, 5.0), [cand(1, 50.0, 5.0 + off(20))], set(), False, CFG)
    assert r['method'] == 'cone' and 'wide_match' in r['flags']
    # cone: close pair -> nearest TARG wins even if the other is brighter
    r = decide(spec(50.0, 5.0, targ=(50.0, 5.0 + off(1))),
               [cand(1, 50.0, 5.0 + off(1), vmag=11.0), cand(2, 50.0, 5.0 - off(5), vmag=5.0)], set(), False, CFG)
    assert r['oid'] == 1 and 'close_pair' in r['flags']
    # cone: equidistant pair -> brightness tie-break
    r = decide(spec(50.0, 5.0), [cand(1, 50.0, 5.0 + off(4), vmag=11.0), cand(2, 50.0, 5.0 - off(4), vmag=5.0)],
               set(), False, CFG)
    assert r['oid'] == 2 and 'close_pair' in r['flags']
    # bare system name resolves to components and is flagged
    names = {1: {canon_name('* alf Cen A')}, 2: {canon_name('* alf Cen B')}, 3: {canon_name('* alf Cen')}}
    oids, amb = unit_name_oids(canon_reduced('alphacen'), [1, 2, 3], names, {3: '**'})
    assert sorted(oids) == [1, 2] and amb
    # an exact star wins over its lettered companion
    hd = {1: {canon_name('HD 102365')}, 2: {canon_name('HD 102365B')}}
    assert unit_name_oids(canon_reduced('hd102365'), [1, 2], hd, {1: 'PM*', 2: '*'}) == ([1], False)
    assert canon_name('* nu. Oct') == canon_name('nu_Oct') and canon_name('* mu. Ara') == canon_name('mu Ara')
    oids, amb = unit_name_oids(canon_reduced('alphacena'), [1, 2, 3], names)
    assert oids == [1] and not amb
    r = decide(spec(50.0, 5.0), [cand(1, 50.0, 5.0 + off(3)), cand(2, 50.0, 5.0 + off(6))], {1, 2}, True, CFG)
    assert r['oid'] == 1 and r['flags'] == ['component_ambiguous']
    # nothing around
    assert decide(spec(50.0, 5.0), [], set(), False, CFG)['method'] == 'unresolved'
    assert nonstellar_id('sun') == 'nonstellar:SUN' and nonstellar_id('ceres') == 'nonstellar:ceres'
    assert nonstellar_id('c/2020f3') and nonstellar_id('hd128620') is None
    print('All decision tests passed')